In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

np.random.seed(42)
tf.random.set_seed(42)

# Downloads once (~170 MB), then cached locally in the runtime
(X_tr_all, y_tr_all), (X_te_all, y_te_all) = keras.datasets.cifar10.load_data()

# Fast subset for quick lab verification (remove [:10000] / [:2000] for full dataset)
X_train_full = X_tr_all[:10000].astype("float32") / 255.0
y_train_full = y_tr_all[:10000].ravel()
X_test = X_te_all[:2000].astype("float32") / 255.0
y_test = y_te_all[:2000].ravel()

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.15, random_state=42, stratify=y_train_full
)

class_names = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

inputs = keras.Input(shape=(32, 32, 3))
x = layers.Conv2D(32, (3, 3), activation="relu", padding="same", name="conv1")(inputs)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.Conv2D(64, (3, 3), activation="relu", padding="same", name="conv2")(x)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.Flatten()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(10, activation="softmax")(x)

cnn = keras.Model(inputs, outputs)
cnn.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

hist = cnn.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=5, batch_size=256, verbose=1)

test_loss, test_acc = cnn.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Accuracy: {test_acc:.4f} | Test Loss: {test_loss:.4f}")

# 1. Accuracy & Loss Curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(hist.history["accuracy"], marker="o", label="Train Acc")
axes[0].plot(hist.history["val_accuracy"], marker="s", label="Val Acc")
axes[0].set_title("CNN Accuracy Curve")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(hist.history["loss"], marker="o", label="Train Loss")
axes[1].plot(hist.history["val_loss"], marker="s", label="Val Loss")
axes[1].set_title("CNN Loss Curve")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
plt.tight_layout()
plt.savefig("exp11_cnn_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# 2. Confusion Matrix & Sample Misclassifications
y_pred = np.argmax(cnn.predict(X_test, verbose=0), axis=1)
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
im = axes[0].imshow(cm, cmap="Blues")
axes[0].set_xticks(range(10))
axes[0].set_xticklabels(class_names, rotation=35, ha="right", fontsize=8)
axes[0].set_yticks(range(10))
axes[0].set_yticklabels(class_names, fontsize=8)
axes[0].set_title("Confusion Matrix")
plt.colorbar(im, ax=axes[0])

mis_idx = np.where(y_pred != y_test)[0][:6]
axes[1].axis("off")
axes[1].set_title("Sample Misclassifications")
inner = fig.add_gridspec(2, 3, left=0.55, right=0.98, top=0.85, bottom=0.12, wspace=0.35, hspace=0.45)
for k, idx in enumerate(mis_idx):
    ax_m = fig.add_subplot(inner[k // 3, k % 3])
    ax_m.imshow(X_test[idx])
    ax_m.set_title(f"T:{class_names[y_test[idx]][:4]}\nP:{class_names[y_pred[idx]][:4]}", fontsize=7.5)
    ax_m.axis("off")
plt.savefig("exp11_cm_and_misclassified.png", dpi=150, bbox_inches="tight")
plt.show()

# 3. Conv1 Feature Maps
fmap_model = keras.Model(inputs=cnn.inputs, outputs=cnn.get_layer("conv1").output)
fmaps = fmap_model.predict(X_test[:1], verbose=0)[0]

fig, axes = plt.subplots(2, 4, figsize=(11, 5.2))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(fmaps[:, :, i], cmap="viridis")
    ax.set_title(f"Filter {i+1}", fontsize=9)
    ax.axis("off")
plt.suptitle("First Conv Layer Feature Maps", fontsize=12)
plt.tight_layout()
plt.savefig("exp11_conv1_feature_maps.png", dpi=150, bbox_inches="tight")
plt.show()

  2670592/170498071 ━━━━━━━━━━━━━━━━━━━━ 28:54 10us/step